# UABR — Phase 2: build `data_v1` (facts, ledger, tokenizer, 1B-token corpus)
**Kaggle settings:** Accelerator **None** (CPU only; saves GPU quota) · Internet **On**.

**How to run (two passes, same code path):**
1. `QUICK = True` → **Run All** interactively (~5–10 min). Every step must succeed and all audits must PASS.
2. `QUICK = False` → **Save Version › Save & Run All (Commit)**. Builds the full 1B-token corpus in the background (expected 1–3 h). You can close the browser.

The output folder `/kaggle/working/data_v1` becomes the input of the Phase 3 pretraining notebook (*Add Input › Your Work › this notebook*).

In [ ]:
QUICK = True            # True = small test run; False = full data_v1 (use Save & Run All)
TARGET_TOKENS = 20_000_000 if QUICK else 1_005_000_000
TOKENIZER_SAMPLE_CHARS = 5_000_000 if QUICK else 40_000_000
REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"

import os, subprocess
WORK = "/kaggle/working" if os.path.exists("/kaggle") else os.getcwd()
DATA = f"{WORK}/data_v1"
REPO_DIR = f"{WORK}/uabr_slm"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
os.makedirs(DATA, exist_ok=True)
print("QUICK =", QUICK, "| target tokens =", f"{TARGET_TOKENS:,}", "| DATA =", DATA)
!git log --oneline -1
!pip install -q datasets tokenizers pyarrow pandas
!python -m pytest tests -q

## 1. Answer vocabularies from Wikidata
Three small SPARQL queries (countries, capitals, languages, occupations). If Wikidata is unreachable the script falls back to built-in lists and records `source: fallback` — that is allowed but must be reported.

In [ ]:
!python -m src.data.vocab --out {DATA}/raw/value_vocab.json

## 2. Controlled facts, exposure tiers, splits, prompts, evidence, ledger
12,000 pseudonymous entities × 3 relations = 36,000 facts; ~50% exposed (tiers 5 / 20 / 80 statements), 50% withheld; split by entity 50/20/10/20.

In [ ]:
!python -m src.data.build_facts --vocab {DATA}/raw/value_vocab.json --out {DATA}/facts

## 3. Instruction-tuning data (Dolly-15k + Alpaca-cleaned)

In [ ]:
!python -m src.data.sft_data --out {DATA}/raw/sft_raw.jsonl

## 4. Tokenizer (byte-level BPE, 12,000 tokens, special tokens fixed at IDs 0–9) + audit
Stops with an error if special tokens, round-trip or the pseudonym leakage scan fail.

In [ ]:
!python -m src.data.tokenizer --facts {DATA}/facts --sft {DATA}/raw/sft_raw.jsonl --out {DATA}/tokenizer --sample_chars {TOKENIZER_SAMPLE_CHARS}

## 5. Pack: Wikipedia + interleaved fact statements → `train.bin`, `val.bin`; SFT → jsonl
Prints progress every 50M tokens.

In [ ]:
!python -m src.data.pack --data {DATA} --sft_raw {DATA}/raw/sft_raw.jsonl --target_tokens {TARGET_TOKENS}

## 6. The six ledger audits (handbook §10.1)

In [ ]:
!python scripts/audit_ledger.py --data {DATA}

## 7. Summary for the experiment sheet

In [ ]:
import json
m = json.load(open(f"{DATA}/manifest.json"))
p, s = m["pretrain"], m["sft"]
tr = json.load(open(f"{DATA}/tokenizer/tokenizer_report.json"))
au = json.load(open(f"{DATA}/audit_report.json"))
print(f"vocab source          : {m['vocab_source']}")
print(f"train tokens          : {p['train_tokens']:,}  (wiki unique {p['wiki_tokens_unique']:,} + statements {p['statement_tokens_repeated']:,})")
print(f"reached target        : {p['reached_target']}  | val tokens {p['val_tokens']:,}  | pack time {p['minutes']} min")
print(f"SFT                   : {s['sft_train_examples']:,} examples, {s['sft_train_tokens']:,} tokens; val {s['sft_val_examples']}")
print(f"tokenizer             : vocab {tr['vocab_size']:,}, chars/token {tr['by_group']}, sha256 {tr['sha256'][:12]}")
print(f"audits                : {au['pass']}")
print("DATA_V1 READY" if au["all_pass"] and (QUICK or p["reached_target"]) else "NOT READY - fix the failing step")
!du -sh {DATA}/*

## 8. Before Phase 3
- Open `data_v1/audit_alias_sample.csv` and check the 100 rows by eye; note the result in the sheet.
- After the **full** run, this notebook's output is `data_v1`. Do not re-run the full build later: the tokenizer and ledger are now frozen.